In [1]:
!nvidia-smi

Mon May 11 16:08:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             26W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /kaggle/working

# 1. IL VERO SEGRETO: Downgrade forzato a PyTorch 2.2.0 (La versione magica dell'autore)
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu121

# 2. Prepariamo il terreno
!apt-get update && apt-get install -y libopenblas-dev
!pip install "numpy<2" "scipy==1.13.1"

# 3. Scarichiamo Minkowski (versione patchata dall'autore per PyTorch 2.2)
!rm -rf MinkowskiEngine
!git clone https://github.com/renezurbruegg/MinkowskiEngine.git
%cd MinkowskiEngine

# 4. Patch per il bug numpy.distutils (esclusivo di Python 3.12, l'autore usava Python 3.10)
patch_code = """
with open('setup.py', 'r') as f:
    text = f.read()

text = text.replace("import numpy.distutils.system_info as sysinfo", "pass")
dummy_sysinfo = '''
class sysinfo:
    @staticmethod
    def get_info(name):
        return {"libraries": ["openblas"], "library_dirs": ["/usr/lib/x86_64-linux-gnu"], "include_dirs": ["/usr/include"]}
'''
text = dummy_sysinfo + '\\n' + text

with open('setup.py', 'w') as f:
    f.write(text)
"""
with open("patch_me.py", "w") as f:
    f.write(patch_code)
!python patch_me.py

# 5. Creiamo IL FILE .WHL (Ora compilerà senza fare storie!)
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda'
!python setup.py bdist_wheel

# 6. Installiamo la nostra build perfetta
!pip install dist/*.whl

print("\n🎉 FATTO! Ricordati a fine sessione di scaricare il file .whl dalla cartella /kaggle/working/MinkowskiEngine/dist/")


/kaggle/working
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 2.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 2.7 MB/s eta 0:00:000:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 90.1 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 73.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 41.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 97.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 13.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 35.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
%cd /kaggle/working

# 1. Scarichiamo il codice dell'autore
!git clone https://github.com/renezurbruegg/icg_net.git
!git clone https://github.com/renezurbruegg/icg_benchmark.git

import os
os.environ['CUDA_HOME'] = '/usr/local/cuda'

# 2. DIPENDENZE PYG (Ora possiamo usare i pacchetti esatti della guida senza crash!)
!pip install torch_geometric
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.2.0+cu121.html

# 3. Dipendenze esterne
!pip install hydra-core open3d trimesh networkx

# 4. PATCH SORGENTE: Aggiorniamo la sintassi deprecata di Hydra
!sed -i 's/from hydra.experimental import compose, initialize/from hydra import compose, initialize/g' /kaggle/working/icg_net/icg_net/icg_net.py
!sed -i 's/with initialize():/with initialize(version_base=None):/g' /kaggle/working/icg_net/icg_net/icg_net.py

# 5. Compiliamo PointNet2
%cd /kaggle/working/icg_net/icg_net/third_party/pointnet2
!pip install .

# 6. PATCH E COMPILAZIONE MCUBES
%cd /kaggle/working/icg_net/icg_net/utils/mcubes
!sed -i 's/"src\/mcubes.pyx"/"src\/mcubes.cpp"/g' setup.py
!sed -i 's/cythonize(ext_modules)/ext_modules/g' setup.py
!python setup.py build_ext --inplace

# 7. Installiamo i requisiti finali
%cd /kaggle/working/icg_net
!pip install -r requirements.txt
%cd /kaggle/working/icg_benchmark
!pip install -e .


/kaggle/working
Cloning into 'icg_net'...
remote: Enumerating objects: 292, done.
remote: Counting objects: 100% (292/292), done.
remote: Compressing objects: 100% (238/238), done.
remote: Total 292 (delta 39), reused 281 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (292/292), 3.80 MiB | 21.24 MiB/s, done.
Resolving deltas: 100% (39/39), done.
Cloning into 'icg_benchmark'...
remote: Enumerating objects: 237, done.
remote: Counting objects: 100% (237/237), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 237 (delta 72), reused 229 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (237/237), 794.91 KiB | 6.52 MiB/s, done.
Resolving deltas: 100% (72/72), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.4 MB/s eta 0:00:00a 0:00:01
Looking in links: https://data.pyg.org/whl/torch-2.2.0+cu121.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 

In [4]:
%cd /kaggle/working/icg_net
import torch
import numpy as np
from icg_net import get_model

try:
    # Richiamiamo la configurazione base di test pre-fornita
    model = get_model("contact_packed_width_eval.yaml", device="cuda")
    print("\n✅ SUCCESSO ASSOLUTO: L'ecosistema ICGNet è completamente operativo sulla GPU!")
except Exception as e:
    import traceback
    print(f"\n❌ ERRORE: Qualcosa è andato storto: {e}")
    traceback.print_exc()


/kaggle/working/icg_net


ModuleNotFoundError: No module named 'MinkowskiEngine.MinkowskiOps'